In [ ]:
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

rf = joblib.load(
    "../Models/random_forest.joblib"
)

X = pd.read_parquet(
    "../Results/X_final.parquet"
)

y = pd.read_parquet(
    "../Results/y_final.parquet"
)["BinaryLabel"]

df = pd.read_parquet(
    "../Results/cleaned_cic_ids2017.parquet"
)

feature_columns = X.columns.tolist()

print("Model loaded.")

# Baseline/New

In [ ]:
baseline_types = [
    "Benign",
    "DoS",
    "DDoS",
    "Portscan"
]

new_types = [
    "Botnet",
    "Bruteforce",
    "Infiltration",
    "WebAttacks"
]

baseline_df = df[
    df["AttackType"].isin(baseline_types)
].copy()

new_df = df[
    df["AttackType"].isin(new_types)
].copy()

print(
    "Baseline samples:",
    len(baseline_df)
)

print(
    "New samples:",
    len(new_df)
)

# Baseline train/test

In [ ]:
X_baseline = baseline_df[feature_columns]
y_baseline = baseline_df["BinaryLabel"]

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_baseline,
    y_baseline,
    test_size=0.20,
    random_state=42,
    stratify=y_baseline
)

# Train baseline model correctly

In [ ]:
rf_monitor = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_monitor.fit(
    X_train_base,
    y_train_base
)

print(
    "Baseline model trained."
)

# Baseline F1

In [ ]:
baseline_pred = rf_monitor.predict(
    X_test_base
)

baseline_f1 = f1_score(
    y_test_base,
    baseline_pred,
    zero_division=0
)

print(
    f"Baseline F1: {baseline_f1:.4f}"
)

# New data F1

In [ ]:
X_new = new_df[feature_columns]
y_new = new_df["BinaryLabel"]

new_pred = rf_monitor.predict(
    X_new
)

new_f1 = f1_score(
    y_new,
    new_pred,
    zero_division=0
)

print(
    f"New Data F1: {new_f1:.4f}"
)

# Performance degradation

In [ ]:
performance_degradation = (
    baseline_f1 - new_f1
) / baseline_f1

print(
    f"Performance degradation: "
    f"{performance_degradation:.4f}"
)

print(
    f"Percentage: "
    f"{performance_degradation * 100:.2f}%"
)

# Model drift alert

In [ ]:
PERFORMANCE_THRESHOLD = 0.10

if performance_degradation >= PERFORMANCE_THRESHOLD:

    performance_alert = True

    print("MODEL PERFORMANCE DRIFT ALERT")

    print(
        f"Performance degradation "
        f"{performance_degradation:.2%} "
        f">= {PERFORMANCE_THRESHOLD:.2%}"
    )

else:

    performance_alert = False

    print("No significant performance degradation.")

# Load data drift result

In [ ]:
drift_df = pd.read_csv(
    "../Results/drift_results.csv"
)

total_features = len(drift_df)

drifted_features = drift_df[
    drift_df["Drift"] == True
]

drift_ratio = (
    len(drifted_features) /
    total_features
)

data_drift_alert = (
    drift_ratio >= 0.20
)

print(
    f"Data drift ratio: {drift_ratio:.2%}"
)

print(
    "Data drift alert:",
    data_drift_alert
)

# Monitoring summary

In [ ]:
monitoring_summary = pd.DataFrame({
    "Metric": [
        "Baseline F1",
        "New Data F1",
        "Performance Degradation",
        "Drift Ratio"
    ],
    "Value": [
        baseline_f1,
        new_f1,
        performance_degradation,
        drift_ratio
    ]
})

display(monitoring_summary)

# Human-in-the-loop

In [ ]:
human_review_required = (
    data_drift_alert
    or performance_alert
)

monitoring_status = {
    "data_drift": bool(data_drift_alert),
    "performance_drift": bool(performance_alert),
    "human_review_required": bool(
        human_review_required
    )
}

monitoring_status

# Decision logic

In [ ]:
if human_review_required:

    print("HUMAN REVIEW REQUIRED")
    print()
    print("Recommended actions:")
    print("1. Security analyst reviews drifted features.")
    print("2. Validate whether new traffic represents real attacks.")
    print("3. Check false positives and false negatives.")
    print("4. Collect newly labelled traffic.")
    print("5. Retrain the model if drift is confirmed.")

else:

    print("Model monitoring status: NORMAL")
    print("Continue monitoring future traffic.")

# Save monitoring result

In [ ]:
monitoring_summary.to_csv(
    "../Results/monitoring_summary.csv",
    index=False
)

print("Monitoring results saved.")